# 02a Byte BPE Generation

## Purpose

This notebook trains the byte-level BPE tokenizer for the rebuilt workflow.

## Input

- `MyDrive/ProjectRoot/data/splits/train.txt`

## Outputs

- `MyDrive/ProjectRoot/tokenizers/byte_bpe/<setting_label>/vocab.json`
- `MyDrive/ProjectRoot/tokenizers/byte_bpe/<setting_label>/merges.txt`
- Hugging Face tokenizer files saved in the same folder
- `tokenizer_config_summary.json`

## Notes to myself

This is the plain baseline tokenizer. I want this one in place first because it gives me the most neutral comparison point for the later tokenizers.

## Setup note

Same pattern again.

- code and notebooks stay in GitHub
- tokenizer artifacts stay in Drive
- Colab pulls the repo at the start
- the final cell syncs the notebook back to GitHub

In [1]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive

# Mount Google Drive so the notebook can read data files and save outputs.
drive.mount('/content/drive')

# Clone the public GitHub repository into the Colab runtime.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists. Pulling latest changes...')

%cd {REPO_DIR}
!git pull origin main --no-edit -q

# Add the repo to the Python path so src/ imports work across notebooks.
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

print('Colab environment ready.')
print(f'Repo directory: {REPO_DIR}')


Mounted at /content/drive
Cloning repository...
/content/glycan-roberta
Colab environment ready.
Repo directory: /content/glycan-roberta


## Path and setting setup

Here I define the training file and the tokenizer setting label. I want the folder name to come directly from the tokenizer hyperparameters so it is obvious what was trained.

In [2]:
# ==============================================================================
# 1. DEFINE THE TRAINING PATHS AND TOKENIZER SETTINGS
# ==============================================================================
PROJECT_ROOT = '/content/drive/MyDrive/ProjectRoot'
TRAIN_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'splits', 'train.txt')

VOCAB_SIZE = 300
MIN_FREQUENCY = 2
SETTING_LABEL = f'v{VOCAB_SIZE}_m{MIN_FREQUENCY}'

TOKENIZER_OUT_DIR = os.path.join(PROJECT_ROOT, 'tokenizers', 'byte_bpe', SETTING_LABEL)
os.makedirs(TOKENIZER_OUT_DIR, exist_ok=True)

print('Training data path:')
print(TRAIN_DATA_PATH)
print('\nTokenizer output directory:')
print(TOKENIZER_OUT_DIR)
print('\nSetting label:')
print(SETTING_LABEL)

if not os.path.exists(TRAIN_DATA_PATH):
    raise FileNotFoundError(f'Training split not found: {TRAIN_DATA_PATH}')

Training data path:
/content/drive/MyDrive/ProjectRoot/data/splits/train.txt

Tokenizer output directory:
/content/drive/MyDrive/ProjectRoot/tokenizers/byte_bpe/v300_m2

Setting label:
v300_m2


## Train the tokenizer

This is the actual byte-BPE training step. I am fixing the vocabulary size and minimum token frequency here so the setting label and output folder match the training choices.

In [3]:
# ==============================================================================
# 2. TRAIN AND SAVE THE BYTE-LEVEL BPE TOKENIZER
# ==============================================================================
import json

from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast

print(f'Training byte-level BPE tokenizer (vocab={VOCAB_SIZE}, min_frequency={MIN_FREQUENCY})...')

tokenizer = ByteLevelBPETokenizer()

tokenizer.train(
    files=[TRAIN_DATA_PATH],
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=['<s>', '<pad>', '</s>', '<unk>', '<mask>']
)

tokenizer.save_model(TOKENIZER_OUT_DIR)

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer._tokenizer,
    bos_token='<s>',
    eos_token='</s>',
    unk_token='<unk>',
    pad_token='<pad>',
    mask_token='<mask>'
)

hf_tokenizer.save_pretrained(TOKENIZER_OUT_DIR)

tokenizer_summary = {
    'tokenizer_family': 'byte_bpe',
    'setting_label': SETTING_LABEL,
    'vocab_size': VOCAB_SIZE,
    'min_frequency': MIN_FREQUENCY,
    'train_data_path': TRAIN_DATA_PATH,
    'tokenizer_output_dir': TOKENIZER_OUT_DIR,
    'saved_files': sorted(os.listdir(TOKENIZER_OUT_DIR)),
}

summary_json_path = os.path.join(TOKENIZER_OUT_DIR, 'tokenizer_config_summary.json')
with open(summary_json_path, 'w', encoding='utf-8') as file:
    json.dump(tokenizer_summary, file, indent=2)

print('Tokenizer training complete.')
print(f'Tokenizer saved to: {TOKENIZER_OUT_DIR}')

Training byte-level BPE tokenizer (vocab=300, min_frequency=2)...
Tokenizer training complete.
Tokenizer saved to: /content/drive/MyDrive/ProjectRoot/tokenizers/byte_bpe/v300_m2


## Quick sanity check

I do not want to do deep analysis here. I just want to confirm the tokenizer loads, has the expected vocabulary size, and can tokenize a couple of example glycans from the training split.

In [4]:
# ==============================================================================
# 3. LOAD THE SAVED TOKENIZER AND INSPECT SAMPLE OUTPUT
# ==============================================================================
import pandas as pd

with open(TRAIN_DATA_PATH, 'r', encoding='utf-8') as file:
    train_sequences = [line.strip() for line in file if line.strip()]

loaded_tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_OUT_DIR)

sample_sequences = train_sequences[:3]
inspection_rows = []

for sample_index, sequence in enumerate(sample_sequences, start=1):
    token_ids = loaded_tokenizer.encode(sequence, add_special_tokens=False)
    tokens = loaded_tokenizer.convert_ids_to_tokens(token_ids)

    inspection_rows.append(
        {
            'sample_index': sample_index,
            'sequence': sequence,
            'num_tokens': len(tokens),
            'tokens': ' | '.join(tokens[:30]),
        }
    )

inspection_df = pd.DataFrame(inspection_rows)
display(inspection_df)

print(f'Loaded vocabulary size: {len(loaded_tokenizer)}')
print(f'Mask token: {loaded_tokenizer.mask_token}')
print(f'Pad token: {loaded_tokenizer.pad_token}')

,sample_index,sequence,num_tokens,tokens
0,1,Galb1-3GlcNAcb1-3(Galb1-4GlcNAcb1-6)Galb1-4Glc,23,Galb | 1 | - | 3 | GlcNAcb | 1 | - | 3 | ( | G...
1,2,Glcb1-4Glcb1-3Glcb1-4Glcb1-4Glcb1-3Glcb1-4Glcb...,29,Glcb | 1 | - | 4 | Glcb | 1 | - | 3 | Glcb | 1...
2,3,Glca1-3Glca1-3Mana1-2Mana1-2Mana1-3(Mana1-2Man...,53,Glca | 1 | - | 3 | Glca | 1 | - | 3 | Mana | 1...


Loaded vocabulary size: 300
Mask token: <mask>
Pad token: <pad>


## Save a small inspection table

This just gives me a lightweight record of the first sanity check without opening the notebook again later.

In [5]:
# ==============================================================================
# 4. SAVE THE INSPECTION OUTPUT
# ==============================================================================
inspection_path = os.path.join(TOKENIZER_OUT_DIR, 'inspection_preview.csv')
inspection_df.to_csv(inspection_path, index=False)

print(f'Inspection preview saved to: {inspection_path}')

Inspection preview saved to: /content/drive/MyDrive/ProjectRoot/tokenizers/byte_bpe/v300_m2/inspection_preview.csv
